# Privacy Policy / Terms and Conditions Summarizer

We will build a bot that will; scrape a website for links, and find the privacy policy and terms and conditions page. 

It will then provide a client brief in laymens terms of what is allowed and how your data is being used.

## The Scraper

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents  # Written by Ed Donner
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
import os
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

There might be a problem with your API key? Please visit the troubleshooting notebook!


## The Bot

In [6]:
links = fetch_website_links("https://google.com")
links

['https://about.google/?fg=1&utm_source=google-AU&utm_medium=referral&utm_campaign=hp-header',
 'https://store.google.com/AU?utm_source=hp_header&utm_medium=google_ooo&utm_campaign=GS100042&hl=en-AU',
 'https://mail.google.com/mail/&ogbl',
 'https://www.google.com/imghp?hl=en&ogbl',
 'https://www.google.com.au/intl/en/about/products',
 'https://accounts.google.com/ServiceLogin?hl=en&passive=true&continue=https://www.google.com/&ec=futura_exp_og_so_72776762_e',
 'https://www.google.com/webhp?hl=en&ictx=0&sa=X&ved=0ahUKEwjKnoywz5-XAxXPdfUHHcRnKx4QPAgF',
 'https://www.google.com/intl/en_au/ads/?subid=ww-ww-et-g-awa-a-g_hpafoot1_1!o2&utm_source=google.com&utm_medium=referral&utm_campaign=google_hpafooter&fg=1',
 'https://www.google.com/services/?subid=ww-ww-et-g-awa-a-g_hpbfoot1_1!o2&utm_source=google.com&utm_medium=referral&utm_campaign=google_hpbfooter&fg=1',
 'https://google.com/search/howsearchworks/?fg=1',
 'https://policies.google.com/privacy?hl=en-AU&fg=1',
 'https://policies.google

In [7]:
def get_privacy_policy_link(links):
    for link in links:
        if "/privacy" in link.lower():
            return link
    return None

def get_terms_and_conditions_link(links):
    for link in links:
        if "/terms" in link.lower() or "/conditions" in link.lower():
            return link
    return None

We want to only get the privacy policy link, and the terms and conditions link.

In [8]:
privacy_policy_link = get_privacy_policy_link(links)
terms_and_conditions_link = get_terms_and_conditions_link(links)
privacy_policy_link, terms_and_conditions_link

('https://policies.google.com/privacy?hl=en-AU&fg=1',
 'https://policies.google.com/terms?hl=en-AU&fg=1')

We give the llm api context.

In [10]:
system_prompt = """
You are a legal expert that comprehensively analyzes privacy policies and terms and conditions of websites. 
You are good at summarizing complex legal texts to a layperson.
Respond in markdown without code blocks.
"""

def get_user_prompt(url):
    links = fetch_website_links(url)
    privacy_policy_link = get_privacy_policy_link(links)
    terms_and_conditions_link = get_terms_and_conditions_link(links)
    user_prompt = f"""
You are looking at url: {url}
Privacy Policy Link: {privacy_policy_link}
Terms and Conditions Link: {terms_and_conditions_link}

Summarize the privacy policy or terms and conditions of this website in a way that is easy for a layperson to understand.
"""
    return user_prompt

Scrap the target website, get the privacy policy and terms links, construct the user prompt.

In [11]:
user_prompt = get_user_prompt("https://google.com")
user_prompt

'\nYou are looking at url: https://google.com\nPrivacy Policy Link: https://policies.google.com/privacy?hl=en-AU&fg=1\nTerms and Conditions Link: https://policies.google.com/terms?hl=en-AU&fg=1\n\nSummarize the privacy policy or terms and conditions of this website in a way that is easy for a layperson to understand.\n'

In [14]:
def summarize_privacy_policy_and_terms(url):
    user_prompt = get_user_prompt(url)
    response = openrouter.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

## Google's Privacy Policy and Terms

In [ ]:
url = "https://google.com"
summarize_privacy_policy_and_terms(url)

### Summary of Google’s Privacy Policy and Terms & Conditions

---

### Google Privacy Policy (https://policies.google.com/privacy?hl=en-AU&fg=1)

**What Information Google Collects:**
- Google collects information you provide directly (e.g., when you create an account, fill out forms).
- They also collect data automatically through your use of their services, like your location, search queries, device information, and cookies.
- Information from third parties might be obtained as well, such as advertisers or partners.

**How Google Uses Your Information:**
- To provide, maintain, and improve their services.
- To personalize your experience, such as showing relevant ads and content.
- To communicate with you, including customer support.
- To protect Google and users from fraud or abuse.
- For research and development to improve their technology.

**Sharing Your Information:**
- Google does not sell your personal information.
- They may share information:
  - With your consent.
  - With affiliated companies within the Google group (like YouTube).
  - With vendors and service providers who help operate services.
  - For legal reasons, like complying with laws or protecting against fraud.
  - When businesses are merged, sold, or acquired.

**Your Controls and Choices:**
- You can manage your privacy settings in your Google Account.
- You can control ad personalization.
- You can review and delete your search and location history.
- You can use privacy tools like Incognito mode for less data collection.

**Security:**
- Google uses strong security measures to protect your data but reminds users that no online service is 100% secure.

**Children’s Privacy:**
- Google’s services are not for children under 13 (or relevant age in your country), and they have special protections in place.

---

### Google Terms & Conditions (https://policies.google.com/terms?hl=en-AU&fg=1)

**Using Google Services:**
- You must follow the law and Google’s policies when using their services.
- You need a Google Account for many services; keep your password secure.
- You must provide accurate information when creating an account.

**What Google Provides:**
- Google delivers services “as is,” without warranties.
- They can modify or discontinue services at any time.

**Your Content & Conduct:**
- You retain ownership of the content you upload.
- By uploading content, you give Google permission to use it to provide the service.
- You agree not to misuse services, such as spamming or uploading harmful content.

**Liability:**
- Google limits their liability for damages resulting from the use of their services.
- They are not responsible for third-party content.

**Termination:**
- Google can suspend or terminate your account if you violate their rules.

**Disputes:**
- Claims related to Google services are generally resolved under California law.
- Any disputes might require binding arbitration rather than court trials.

---

### In Simple Terms:
- **Google collects various kinds of info about you** to make their services work better and show you tailored content.
- **You have control over what data Google uses,** and you can manage privacy settings.
- **When you use Google, you agree to follow their rules** and not misuse the services.
- **Google limits their liability** and can suspend your account if necessary.
- They aim to keep your data secure but warn that no system is perfectly safe.

If privacy or legal matters are important to you, it’s good practice to regularly check Google’s policies for updates.

## Apple's Privacy Policy and Terms

In [17]:
url = "https://apple.com"
summarize_privacy_policy_and_terms(url)

Here's an easy-to-understand summary of Apple's Privacy Policy and Terms and Conditions based on the links you provided:

---

### Apple Privacy Policy Summary

- **Data Collection**: Apple collects personal information like your name, contact details, and device information when you use their services or products.
- **How Data is Used**: They use your data to provide and improve their services, for customer support, marketing (with your permission), and to comply with legal requirements.
- **Data Sharing**: Apple may share your information with trusted partners and service providers but doesn’t sell your personal information to advertisers.
- **Your Control**: You have options to access, update, or delete your personal data, and you can manage your privacy settings on Apple's platforms.
- **Security**: Apple implements strong security measures to protect your information from unauthorized access.
- **Cookies & Tracking**: They use cookies and similar technology to enhance your experience, but you can control or block these through your browser settings.
- **Children’s Privacy**: Apple takes extra care regarding children’s data and complies with applicable child privacy laws.
- **Cross-Border Data**: Since Apple operates globally, your data may be transferred and stored in different countries, but it's still protected according to their standards.

---

### Apple Terms and Conditions Summary

- **Acceptance**: By using Apple’s website and services, you agree to follow their terms.
- **Service Use**: You agree to use the site legally and not misuse or interrupt services.
- **Content Ownership**: Apple owns the content on their site (like text, graphics, and software) or has licensed it. You can view or download it for personal use, but not reuse or distribute it without permission.
- **Purchases**: If you buy products or services from Apple, the purchase is subject to Apple’s sale policies, including payment, delivery, and return terms.
- **Liabilities**: Apple limits their liability for issues arising from the use of their website or services and disclaims warranties to the extent allowed by law.
- **Changes**: Apple can update the terms and conditions at any time, and continued use of the site means you accept those changes.
- **Governing Law**: Any disputes will be handled according to the laws specified in the terms (usually California law).

---

**In short**: Apple values your privacy, uses your data responsibly, and sets clear rules for how you can use their website and services. You should review the full policies for detailed information, but overall, Apple aims to protect users while providing a secure and legal service experience.

## Microsoft's Privacy Policy and Terms

In [18]:
url = "https://microsoft.com"
summarize_privacy_policy_and_terms(url)

Here’s an easy-to-understand summary of Microsoft’s Privacy Policy from https://privacy.microsoft.com/en-us:

### What Information Microsoft Collects
- **Personal info you give them:** Like your name, email, phone number, payment info, etc., when you create an account or buy products.
- **Info from your use:** Things like the device you use, IP address, browser type, location data, and how you interact with Microsoft services.
- **Content and communications:** Files you upload, emails, chats, and other communications you have through Microsoft services.
- **Data from other sources:** Like public databases, partners, or other services you use connected to Microsoft.

### How Microsoft Uses Your Info
- To provide, improve, and personalize their products and services.
- To communicate with you about updates, offers, or customer support.
- To enhance security and protect against fraud.
- To comply with legal obligations.

### Sharing Your Information
- With your consent.
- With Microsoft affiliates and subsidiaries.
- With trusted partners who help provide services (like data hosting or customer support).
- When legally required (e.g., to law enforcement).
- In case of business changes, like mergers or sales.

### Your Control Over Your Data
- You can access and update your info through your Microsoft account.
- You can manage privacy settings for ads, location, and data collection.
- You can request to delete certain personal data.
- You can opt-out of some marketing communications.

### Data Security and Storage
- Microsoft uses strong security measures to protect your data.
- Your data may be stored and processed in different countries.
- They try to keep data only as long as necessary.

### Kids’ Privacy
- Certain rights and protections for children under 13 or applicable local age.

---

### Summary
Microsoft collects info you provide and data from your use of their products to improve and personalize services. They keep your data safe but may share it with affiliates, partners, or legally required parties. You have control over your personal data and can manage preferences through your account. 

This summary covers key points, but if you want details, you might want to read the full policy at https://privacy.microsoft.com/en-us.

## Mental Health Australia

In [20]:
url = "https://www.mentalhealthaustralia.org.au/"
summarize_privacy_policy_and_terms(url)

Here’s an easy-to-understand summary of the **Privacy Policy** for the website of Mental Health Australia (based on their /privacy-policy page):

---

### What Information They Collect
- They collect personal details you provide, like your name, email, phone number, or any other info you share when you contact them or sign up for newsletters.
- They might also collect information automatically, like your IP address and how you use their website (through cookies or similar technology).

### How They Use Your Information
- To respond to your queries or provide the services you asked for.
- To send you updates, newsletters, or information related to mental health if you subscribed.
- To improve their website and services by understanding how people use it.
- To comply with legal obligations.

### Sharing Your Information
- They generally do **not** sell or rent your data.
- They may share information with trusted service providers who help them run the website or provide services (like email providers).
- They might disclose your information if required by law or for safety reasons.

### Protecting Your Information
- They take reasonable steps to keep your personal information safe from unauthorized access, misuse, or disclosure.
- However, they can’t guarantee 100% security, especially over the internet.

### Your Choices and Rights
- You can choose to opt-out of receiving marketing communications at any time.
- You can ask to access, correct, or delete your personal information they hold.
- You can manage cookie preferences through your browser settings.

### Cookies and Tracking
- They use cookies to improve user experience and analyze how visitors use the website.
- You can disable cookies in your browser, but some website functions might not work properly.

### Contact
- If you have questions about your privacy or want to make a request about your personal data, you can contact them through the details provided on their website.

---

**In short:** Mental Health Australia collects information you provide and some data automatically to help them improve their website and communicate with you. They protect your information responsibly, don’t sell it, and respect your rights about your personal data. If you want to know more or manage your info, you can get in touch with them directly.